<h3>실습 안내</h3>
<p>프로젝트에 제공된 <code>E01.pdf</code>는 수업을 위해 내용을 지어 만든 문서입니다. 파일을 복사할 필요는 없습니다. 코드가 현재 폴더에서 시작해 부모 폴더를 하나씩 확인하고 원본 위치를 찾습니다.</p>
<h3>첫 실행 코드를 읽는 순서</h3><p><code>PdfReader</code>와 <code>Document</code>, 스플리터, 벡터스토어는 문서가 바뀔 때 실행하는 색인 단계입니다. 질문이 들어오면 <code>similarity_search_with_score()</code>가 청크와 거리를 반환하고, 그 본문을 프롬프트에 넣은 뒤 <code>model.invoke()</code>를 호출할 때 생성이 실행됩니다.</p><p>같은 PDF에서 질문만 바뀌면 질문 처리를 다시 실행할 수 있지만, PDF 본문이나 임베딩 모델이 바뀌면 색인부터 다시 만들어야 합니다. 마지막 답이 이상하면 검색된 청크와 모델 답을 따로 확인합니다.</p>

In [1]:
# PDF, 텍스트 분할, 임베딩, 생성에 필요한 패키지를 설치합니다.
# ==로 버전을 고정해 모든 학생이 같은 API를 사용하게 합니다.
%pip install -q "pypdf==6.17.0" "langchain==1.4.0" "langchain-openai==1.6.0" "langchain-text-splitters==1.1.2" "python-dotenv==1.2.3"

Note: you may need to restart the kernel to use updated packages.


<h3>1. 제공된 PDF 찾기</h3>
<p>노트북을 어느 폴더에서 열었든 PDF를 찾을 수 있도록 현재 폴더부터 부모 폴더를 하나씩 확인합니다.</p>

In [2]:
# start는 현재 작업 폴더이고 project_root는 찾은 프로젝트 루트입니다.
# parents는 부모 폴더들을 가까운 순서로 제공합니다.
from pathlib import Path

start = Path.cwd().resolve()
project_root = None
folders = [start]
folders.extend(start.parents)
for folder in folders:
    corpus = folder / "labs" / "data" / "rag-corpus"
    if corpus.exists():
        project_root = folder
        break
assert project_root is not None, "labs/data/rag-corpus를 찾지 못했습니다."
pdf_path = project_root / "labs/data/rag-corpus/pdf/easy/E01.pdf"
assert pdf_path.exists(), f"PDF가 없습니다: {pdf_path}"
print("PDF:", pdf_path.name)

PDF: E01.pdf


<h3>2. PDF에서 쪽별 텍스트 읽기</h3>
<p><code>PdfReader</code>가 PDF를 읽으면 <code>pages</code>에 쪽별 데이터가 들어갑니다. 각 쪽을 <code>Document</code>로 바꿉니다. Document는 본문과 출처 정보를 한 묶음으로 담는 LangChain 객체입니다.</p>

In [3]:
# extract_text()는 한 쪽의 텍스트를 문자열로 반환합니다.
# enumerate(..., start=1)로 사람이 읽는 쪽 번호를 metadata에 저장합니다.
from pypdf import PdfReader
from langchain_core.documents import Document

reader = PdfReader(pdf_path)
pages = []
for page_number, page in enumerate(reader.pages, start=1):
    page_text = page.extract_text()
    document = Document(
        page_content=page_text,
        metadata={"doc_id": "E01", "page": page_number, "source": pdf_path.name},
    )
    pages.append(document)
print("Document 수:", len(pages), "타입:", type(pages[0]).__name__)
print(pages[1].page_content[:180])

Document 수: 5 타입: Document
1. 도서관 소개
미르시립도서관은 1998년 10월 미르시 중앙동에 문을 연 공공도서관입니다. 지상 4층, 지하 1층 규모
이며 장서 42만 권과 전자책 6만 종을 보유하고 있습니다. 2024년 리모델링을 마치고 3층에 '메이커 스
페이스'와 '조용한 열람실'을 새로 열었습니다.
도서관의 사명은 "누구나, 언제나, 무료로


<h3>3. 쪽을 검색 단위로 나누기</h3>
<p><code>RecursiveCharacterTextSplitter</code>는 쪽 본문을 검색하기 좋은 작은 조각인 청크로 나눕니다. 문서 ID와 쪽 번호 같은 출처 정보(metadata)도 각 청크에 복사합니다. API 비용을 낮추기 위해 질문과 관련된 2~3쪽만 숫자 벡터로 바꿉니다.</p>

In [4]:
# chunk_size는 청크 최대 길이, chunk_overlap은 이웃 청크의 공유 길이입니다.
# split_documents()는 Document 리스트를 받고 더 작은 Document 리스트를 반환합니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
practice_pages = pages[1:3]
chunks = splitter.split_documents(practice_pages)
for chunk_number, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_number
print("청크 수:", len(chunks), "첫 길이:", len(chunks[0].page_content))
print("첫 metadata:", chunks[0].metadata)

청크 수: 4 첫 길이: 498
첫 metadata: {'doc_id': 'E01', 'page': 2, 'source': 'E01.pdf', 'chunk_id': 0}


<h3>4. API 키 준비</h3>
<p>프로젝트 루트의 <code>.env</code>를 읽고 임베딩과 생성 호출을 준비합니다.</p>

In [5]:
# find_dotenv(usecwd=True)는 현재 작업 폴더를 기준으로 .env 경로를 찾습니다.
# assert는 파일이나 API 키가 없을 때 외부 호출 전에 멈춥니다.
import os
from dotenv import find_dotenv, load_dotenv

env_file = find_dotenv(usecwd=True)
assert env_file, "프로젝트 루트의 .env를 찾지 못했습니다."
load_dotenv(env_file)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 없습니다."
print("API 키 준비 완료")

API 키 준비 완료


<h3>5. 청크를 임베딩해 저장하기</h3>
<p>임베딩 모델은 청크의 뜻을 비교할 수 있도록 글을 숫자 목록인 벡터로 바꿉니다. <code>InMemoryVectorStore</code>는 이 벡터와 원본 Document를 현재 파이썬 실행 메모리에 보관합니다.</p>

In [6]:
# embedding은 사용할 OpenAI 임베딩 모델 객체입니다.
# from_documents()는 임베딩을 실제로 호출하고 검색 가능한 store를 반환합니다.
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(model="text-embedding-3-small")
store = InMemoryVectorStore.from_documents(chunks, embedding)
print("store 타입:", type(store).__name__)

store 타입: InMemoryVectorStore


<h3>6. 질문과 가까운 청크 검색하기</h3>
<p>질문을 임베딩한 뒤 가장 가까운 청크 2개를 가져옵니다. 답을 만들기 전에 문서 ID, 쪽, 본문을 먼저 보여 줍니다.</p>

In [7]:
# hits의 각 요소는 검색된 Document이며 k=2는 상위 2건을 의미합니다.
# 생성 전에 중간 결과를 출력하면 검색 실패와 생성 실패를 구분할 수 있습니다.
question = "미르시립도서관의 정기 휴관일은 언제인가요?"
hits = store.similarity_search(question, k=2)
print("hits 타입:", type(hits).__name__, "개수:", len(hits))
for hit in hits:
    print("---")
    print("metadata:", hit.metadata)
    print("content:", hit.page_content[:220])

hits 타입: list 개수: 2
---
metadata: {'doc_id': 'E01', 'page': 2, 'source': 'E01.pdf', 'chunk_id': 0}
content: 1. 도서관 소개
미르시립도서관은 1998년 10월 미르시 중앙동에 문을 연 공공도서관입니다. 지상 4층, 지하 1층 규모
이며 장서 42만 권과 전자책 6만 종을 보유하고 있습니다. 2024년 리모델링을 마치고 3층에 '메이커 스
페이스'와 '조용한 열람실'을 새로 열었습니다.
도서관의 사명은 "누구나, 언제나, 무료로"입니다. 시민 모두가 자료를 자유롭게 이용하고 배울 수 있도
록 돕는 
---
metadata: {'doc_id': 'E01', 'page': 2, 'source': 'E01.pdf', 'chunk_id': 1}
content: 구역 평일 주말 비고
종합자료실(2층) 09:00~22:00 09:00~18:00
어린이자료실(1층) 09:00~18:00 09:00~18:00 보호자 동반 권장
디지털자료실(2층) 09:00~21:00 09:00~18:00 1회 2시간
조용한 열람실(3층) 07:00~22:00 07:00~18:00 좌석 예약제
메이커 스페이스(3층) 13:00~20:00 13:00~17:00 사전 예약
카


<h3>7. k만 1로 바꾸어 비교</h3>
<p>질문·임베딩·문서는 그대로 두고 검색 개수만 2에서 1로 줄입니다. 생성 전에 출처 후보가 어떻게 달라지는지 확인합니다.</p>

In [8]:
# hits_k1은 같은 question에서 k만 1로 바꾼 Document 리스트입니다.
# 두 반복문으로 k=2와 k=1의 출처 태그를 모아 나란히 출력합니다.
ids_k2 = []
for hit in hits:
    ids_k2.append(f"{hit.metadata['doc_id']} p.{hit.metadata['page']}")
hits_k1 = store.similarity_search(question, k=1)
ids_k1 = []
for hit in hits_k1:
    ids_k1.append(f"{hit.metadata['doc_id']} p.{hit.metadata['page']}")
print("k=2:", ids_k2)
print("k=1:", ids_k1)
assert len(hits_k1) == 1

k=2: ['E01 p.2', 'E01 p.2']
k=1: ['E01 p.2']


<h3>8. 검색 결과로 프롬프트 만들기</h3>
<p>각 청크 앞에 <code>[문서 ID p.쪽]</code> 모양의 출처 태그를 붙입니다. 그런 다음 모델이 참고할 문맥(<code>context</code>) 문자열 하나로 연결합니다.</p>

In [9]:
# context_parts는 각 검색 결과를 별도의 문자열로 담는 리스트입니다.
# join()은 리스트의 문자열을 빈 줄 하나로 구분해 context로 반환합니다.
context_parts = []
for hit in hits:
    doc_id = hit.metadata["doc_id"]
    page = hit.metadata["page"]
    source = f"[{doc_id} p.{page}]"
    context_parts.append(f"{source} {hit.page_content}")
context = "\n\n".join(context_parts)
print(context[:500])

[E01 p.2] 1. 도서관 소개
미르시립도서관은 1998년 10월 미르시 중앙동에 문을 연 공공도서관입니다. 지상 4층, 지하 1층 규모
이며 장서 42만 권과 전자책 6만 종을 보유하고 있습니다. 2024년 리모델링을 마치고 3층에 '메이커 스
페이스'와 '조용한 열람실'을 새로 열었습니다.
도서관의 사명은 "누구나, 언제나, 무료로"입니다. 시민 모두가 자료를 자유롭게 이용하고 배울 수 있도
록 돕는 것을 최우선으로 합니다.
2. 이용 시간
평일(화~금) 이용 시간은 오전 9시부터 오후 10시까지입니다. 토요일과 일요일은 오전 9시부터 오후 6
시까지 문을 엽니다.
매주 월요일은 정기 휴관일 입니다. 법정 공휴일에도 휴관하며, 공휴일이 월요일과 겹치면 그 다음 날인
화요일에도 휴관합니다.
어린이자료실(1층)은 요일과 관계없이 오후 6시에 문을 닫습니다. 디지털자료실(2층)은 오후 9시까지
이용할 수 있습니다.
구역 평일 주말 비고
종합자료실(2층) 09:00~22:00 09:


<h3>9. LangChain으로 답변 생성하기</h3>
<p>삼중따옴표 문자열은 여러 줄 프롬프트를 코드 모양 그대로 적을 때 사용합니다. 문서만 근거로 사용하고 출처를 붙이라고 명시합니다.</p>

In [10]:
# f 삼중 문자열은 context와 question 변수를 여러 줄 템플릿에 삽입합니다.
# invoke()는 AIMessage를 반환하고 .text는 생성된 답변 문자열입니다.
from langchain.chat_models import init_chat_model

prompt = f"""다음 문서만 근거로 질문에 답하세요.
답 끝에 [문서ID p.쪽] 형식의 출처를 붙이세요.

[문서]
{context}

[질문]
{question}
"""
model = init_chat_model(
    "openai:gpt-5.4-mini",
    temperature=0,
    reasoning_effort="none",
    max_tokens=300,
)
answer = model.invoke(prompt)
print(answer.text)

미르시립도서관의 정기 휴관일은 **매주 월요일**입니다. [E01 p.2]


<h3>10. 첫 RAG 완주 확인</h3>
<p>출력 문자열에 PDF의 휴관일과 출처 형식이 함께 있는지 검사합니다.</p>

In [11]:
# in은 특정 문자열이 answer.text 안에 포함되었는지 검사합니다.
# 답과 출처를 따로 검사해 어느 조건이 깨졌는지 바로 알 수 있습니다.
assert "월요일" in answer.text
assert "[E01 p." in answer.text
print("검증 통과: PDF → 분할 → 임베딩 → 검색 → 생성을 완주했습니다.")

검증 통과: PDF → 분할 → 임베딩 → 검색 → 생성을 완주했습니다.


<h3>정리</h3>
<p>수업용 PDF를 쪽별 Document로 읽고 작은 청크로 나눴습니다. 청크를 벡터스토어에 저장하고 검색한 문맥을 모델에 전달해 출처가 있는 답을 만들었습니다.</p>

<p><strong>원인</strong> — 질문만 모델에 전달하면 제공 PDF의 본문과 출처를 사용할 수 없습니다.</p>

In [12]:
# before_prompt는 정상 프롬프트와 지시·질문·출처 형식이 같고 문서 내용만 비어 있습니다.
# 출처가 있어야 한다는 assert는 실패하고 except가 그 사실을 출력합니다.
empty_context = ""
before_prompt = f"""다음 문서만 근거로 질문에 답하세요.
답 끝에 [문서ID p.쪽] 형식의 출처를 붙이세요.

[문서]
{empty_context}

[질문]
{question}
"""
plain_answer = model.invoke(before_prompt)
print(plain_answer.text)
try:
    assert "[E01 p." in plain_answer.text
except AssertionError:
    print("예상한 실패: PDF 문맥이 전달되지 않았습니다.")

제공된 문서에는 미르시립도서관의 정기 휴관일 정보가 없습니다. 따라서 문서만으로는 확인할 수 없습니다.
예상한 실패: PDF 문맥이 전달되지 않았습니다.


<p><strong>수정</strong> — 검색한 PDF 문맥과 출처 형식을 프롬프트에 넣어 같은 질문을 다시 실행합니다.</p>

In [13]:
# fixed_answer는 앞에서 만든 prompt, 즉 PDF 검색 문맥이 포함된 입력의 응답입니다.
# 동일한 휴관일 질문에 답과 E01 출처가 함께 있는지 확인합니다.
fixed_answer = model.invoke(prompt)
print(fixed_answer.text)
assert "월요일" in fixed_answer.text
assert "[E01 p." in fixed_answer.text

미르시립도서관의 정기 휴관일은 **매주 월요일**입니다. [E01 p.2]
